# 🏢 [Lab 2] 기업용 지식 그래프(Knowledge Graph) & Microsoft GraphRAG 구축 및 추론 실무

> **핵심 학습 목표**:
> 1. **Vector RAG vs Graph RAG의 본질적 차이 체감**: 단순 벡터 검색이 다중 홉(Multi-hop) 조직 결재선 추적 및 전사적 거시 질문(Global Sensemaking)에서 실패하는 원인을 규명합니다.
> 2. **Part 1 [기초 Graph RAG 직접 코딩]**: Pydantic Structured Output 기반 트리플렛(`(Subject, Relation, Object)`) 추출 $\rightarrow$ NetworkX `DiGraph` 모델링 $\rightarrow$ **PyVis 인터랙티브 시각화** $\rightarrow$ **1-hop vs N-hop 서브그래프 탐색** 파이프라인을 직접 코딩합니다.
> 3. **Part 2 [Microsoft GraphRAG 구축 원리 & 사전 구축 데이터 기반 추론]**: Leiden 알고리즘 기반 계층 커뮤니티 클러스터링 및 커뮤니티 리포트 생성 원리를 학습하고, **사전 구축된 GraphRAG 산출물(Parquet & LanceDB)**을 기반으로 **Global Search(Map-Reduce 전사 종합)**와 **Local Search(엔티티 주변부 정밀 탐색)**를 실행합니다.
> 4. **Part 3 [LangGraph 지능형 에이전트 도구 바인딩]**: 에이전트가 질문 의도(거시 요약 vs 미시 관계)에 따라 최적의 지식 그래프 도구를 자율 선택(Routing)하여 최종 답변을 도출하는 Agentic RAG를 완성합니다.

---

### 🗺️ 지식 그래프(Knowledge Graph) & GraphRAG 2-Track 아키텍처

```
[Track 1: 기초 Graph RAG 원리 (직접 코딩)]       [Track 2: MS GraphRAG 실무 (공식 패키지)]
┌──────────────────────────────────────┐        ┌──────────────────────────────────────┐
│ • 사내 조직도/프로젝트 원천 텍스트    │        │ • 엔터프라이즈 통합 지식 코퍼스      │
│ • Pydantic Structured Output 추출    │        │ • graphrag init & settings.yaml      │
│ • NetworkX DiGraph & PyVis HTML 시각화│        │ • Leiden 커뮤니티 계층 클러스터링     │
│ • 1-hop vs 2-hop BFS 서브그래프 탐색 │        │ • Global Search (Map-Reduce 거시)    │
│ • LLM 컨텍스트 주입 & 추론           │        │ • Local Search (엔티티 중심 미시)    │
└──────────────────┬───────────────────┘        └──────────────────┬───────────────────┘
                   │                                               │
                   └───────────────────────┬───────────────────────┘
                                           ▼
                      [LangGraph 에이전트 도구 바인딩 & 자율 라우팅]
```


## 1. 환경 설정 및 Vertex AI Gemini 연결

### 💡 엔터프라이즈 Graph RAG를 위한 핵심 라이브러리 구성
* `langchain-google-vertexai`: Google Cloud Vertex AI 기반 고성능 `gemini-3.7-flash` 및 `text-embedding-004` 연동
* `pydantic`: 정밀한 Entity-Relation-Entity 구조화 출력 스키마 정의
* `networkx`: 인메모리 방향성 지식 그래프(DiGraph) 모델링 및 BFS N-hop 순회
* `pyvis`: 브라우저에서 드래그/호버/줌이 가능한 인터랙티브 HTML 물리 네트워크 시각화
* `graphrag`: Microsoft 공식 계층 커뮤니티 GraphRAG 엔진 (Leiden Clustering & Map-Reduce Query)


In [ ]:
import os
import sys
from dotenv import load_dotenv

# 프로젝트 루트를 sys.path에 추가
PROJECT_ROOT = os.path.abspath('..') if os.path.exists('../rag') else os.path.abspath('.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

ENV_PATH = os.path.join(PROJECT_ROOT, '.env')
if os.path.exists(ENV_PATH):
    load_dotenv(ENV_PATH, override=True)
    print(f'✅ 환경변수 로드 완료: {ENV_PATH}')
else:
    load_dotenv(override=True)


## 2. 사내 조직도 및 전략 프로젝트 원천 데이터 로드

복합적인 관계 추론이 필요한 엔터프라이즈 데이터(`data/graph/org_relationships.txt`)를 준비합니다.
- **3대 사업본부 및 산하 6개 팀 조직 구조 & 전결 위임 규정**
- **3대 핵심 전략 프로젝트(P-01, P-02, P-03) 총괄 PM, 투입 인력 및 승인 예산**
- **부서 간 협업/인력 파견 및 프로젝트 간 기술 의존성 관계**


In [ ]:
DATA_PATH = "data/graph/org_relationships.txt"
if not os.path.exists(DATA_PATH):
    DATA_PATH = "../data/graph/org_relationships.txt"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_org_text = f.read()

print("=" * 70)
print(f"📄 [사내 원천 데이터 확인: {len(raw_org_text)}자]")
print("=" * 70)
print(raw_org_text[:500] + "\n... (중략) ...\n" + raw_org_text[-350:])


## 3. [문제 제기] Vector RAG(순수 벡터 검색)의 다중 홉 한계 체감

### ❓ 검증 질문:
> *"클라우드운영팀 김철수 수석이 총괄하는 프로젝트의 예산을 최종 승인하는 본부장은 누구인가?"*

### 🧠 필요한 다중 홉(Multi-Hop) 추론 경로:
1. **Hop 1**: `[클라우드운영팀 김철수 수석]` $\rightarrow$ 총괄 프로젝트는 `[P-01: 클라우드 네이티브 마이그레이션 (예산 3.5억)]`
2. **Hop 2**: 5,000만원 초과 프로젝트 예산 규정에 따라 소속 본부장인 `[클라우드사업본부 박영희 전무]`가 최종 승인권자임

단순 청킹된 Vector RAG는 직급 체계 청크와 프로젝트 청크가 서로 다른 청크로 분리되어 있어 이 관계를 연결하지 못하고 실패합니다.


## 4. [Part 1] Pydantic Structured Output을 활용한 지식 그래프(Triplet) 직접 추출

LLM에게 엄격한 Pydantic 스키마(`Triplet`, `KnowledgeGraph`)를 부여하여, 텍스트에서 비정형 관계를 배제하고 정확한 `(Subject, Relation, Object)` 트리플렛 목록을 구조화 추출합니다.


In [ ]:
from typing import List
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

# 1. Pydantic 스키마 정의
class Triplet(BaseModel):
    subject: str = Field(description="관계의 주체(출발 노드 엔티티). 고유명사나 명사형이어야 합니다. 예: 클라우드사업본부")
    relation: str = Field(description="두 엔티티 간의 구체적 관계(엣지 라벨). 예: 본부장, 소속팀원, 총괄PM, 예산승인권자")
    object: str = Field(description="관계의 대상(도착 노드 엔티티). 예: 박영희 전무, 김철수 수석")

class KnowledgeGraph(BaseModel):
    triplets: List[Triplet] = Field(description="문서에서 추출된 정밀한 지식 그래프 트리플렛 목록")

llm = init_chat_model(model="gemini-3.7-flash", model_provider="google_genai")

# 2. Structured Output 바인딩
structured_llm = llm.with_structured_output(KnowledgeGraph)

extraction_prompt = f"""당신은 엔터프라이즈 지식 그래프 구축 전문가입니다.
아래 제공된 [사내 원천 텍스트]를 면밀히 분석하여, 조직 구조, 직급, 결재 전결 규정, 전략 프로젝트, PM, 투입 인력, 예산, 의존성 관계를 모두 포괄하는 지식 그래프 트리플렛(Subject, Relation, Object)을 빠짐없이 추출하세요.

[사내 원천 텍스트]:
{raw_org_text}
"""

print("⏳ LLM을 활용한 트리플렛 구조화 추출 실행 중...")
kg_result: KnowledgeGraph = structured_llm.invoke(extraction_prompt)

print(f"\n✅ 추출 완료! 총 {len(kg_result.triplets)}개의 트리플렛이 생성되었습니다.")
print("=" * 70)
for i, t in enumerate(kg_result.triplets[:10], 1):
    print(f"{i:02d}. [{t.subject}] ───({t.relation})───> [{t.object}]")
    
print(f"... (총 {len(kg_result.triplets)}개 트리플렛 구축 완료)")

## 5. [Part 1] NetworkX 지식 그래프 모델링 & PyVis 인터랙티브 시각화

추출된 트리플렛을 NetworkX `DiGraph`(방향성 그래프)로 적재하고, 브라우저에서 드래그/호버/줌이 가능한 **PyVis 인터랙티브 HTML 그래프**로 렌더링합니다.


In [ ]:
import networkx as nx
from pyvis.network import Network
import IPython.display

# 1. NetworkX DiGraph 생성
G = nx.DiGraph()
for t in kg_result.triplets:
    G.add_edge(t.subject.strip(), t.object.strip(), relation=t.relation.strip())

print("=" * 70)
print("📊 [지식 그래프 모델링 결과]")
print(f"  • 총 엔티티 (노드 수): {G.number_of_nodes()}개")
print(f"  • 총 관계 (엣지 수): {G.number_of_edges()}개")
print("=" * 70)

# 2. PyVis 시각화 생성 (CDN 리소스 모드로 완전 독립형 HTML 생성)
net = Network(height="550px", width="100%", bgcolor="#1a1b26", font_color="#ffffff", directed=True, cdn_resources="remote")

for node in G.nodes():
    if "본부" in node or "팀" in node:
        color, size = "#7aa2f7", 25  # 블루 (조직/부서)
    elif any(pos in node for pos in ["전무", "상무", "수석", "책임", "선임", "팀장", "CFO"]):
        color, size = "#9ece6a", 22  # 그린 (인물/직급)
    elif "P-0" in node or "프로젝트" in node:
        color, size = "#f7768e", 28  # 레드 (프로젝트)
    else:
        color, size = "#e0af68", 18  # 옐로우 (규정/예산/기타)
    net.add_node(node, label=node, color=color, size=size)

for u, v, data in G.edges(data=True):
    net.add_edge(u, v, label=data.get("relation", "관련"), color="#565f89", arrows="to")

html_file = "enterprise_knowledge_graph.html"
html_content = net.generate_html()
with open(html_file, "w", encoding="utf-8") as f:
    f.write(html_content)
print(f"✅ PyVis 인터랙티브 그래프가 '{html_file}' 파일로 저장되었습니다.")

# 주피터 노트북 인라인 IFrame 렌더링
IPython.display.IFrame(src=html_file, width="100%", height="580px")


### 🔍 [Part 1] 1-Hop vs 2-Hop 서브그래프 다중 홉 추론 비교

질문 엔티티(`김철수 수석`, `클라우드운영팀`)에서 시작하여 BFS(너비 우선 탐색)로 서브그래프를 확장했을 때의 추론 성공 여부를 비교합니다.

```mermaid
graph LR
    A["김철수 수석"] -->|소속팀장| B["클라우드운영팀"]
    A -->|총괄PM| C["P-01 클라우드 마이그레이션"]
    B -->|소속본부| D["클라우드사업본부"]
    D -->|본부장| E["★ 박영희 전무 (최종 예산 승인권자)"]
    
    style A fill:#9ece6a,stroke:#333,stroke-width:2px,color:#000
    style C fill:#f7768e,stroke:#333,stroke-width:2px,color:#000
    style E fill:#ff9e3b,stroke:#333,stroke-width:3px,color:#000



In [ ]:
from rag.networkx_graph import multi_hop_search
from utils.message_utils import normalize_content

target_query = "클라우드운영팀 김철수 수석이 총괄하는 프로젝트의 최종 예산 승인권자(본부장)는 누구인가요?"

print("=" * 70)
print(f"🔍 [사용자 질문]: {target_query}")
print("=" * 70)

# 1. 1-Hop 서브그래프 검색
sub_1hop, visited_1hop = multi_hop_search(G, seed_entities=["김철수 수석", "클라우드운영팀"], max_hops=1)
print(f"\n🔹 [1-Hop 검색된 서브그래프 컨텍스트] (방문 엔티티: {len(visited_1hop)}개):")
for edge in sub_1hop[:4]:
    print(f"  - [{edge[0]}] --({edge[2]['relation']})--> [{edge[1]}]")

# 2. 2-Hop 서브그래프 검색 (다중 홉 확장)
sub_2hop, visited_2hop = multi_hop_search(G, seed_entities=["김철수 수석", "클라우드운영팀"], max_hops=2)
print(f"\n----------------------------------------------------------------------")
print(f"🔸 [2-Hop 검색된 서브그래프 컨텍스트] (방문 엔티티: {len(visited_2hop)}개):")
for edge in sub_2hop[:6]:
    print(f"  - [{edge[0]}] --({edge[2]['relation']})--> [{edge[1]}]")

# 3. 2-Hop 지식 그래프 컨텍스트를 LLM에 주입하여 추론
graph_context = "\n".join([f"- [{u}] --({d['relation']})--> [{v}]" for u, v, d in sub_2hop])
# graph_prompt = f"""당신은 사내 지식 그래프 추론 AI입니다. 아래 제공된 [지식 그래프 관계 데이터]만을 바탕으로 질문에 대한 단계별 추론 경로와 최종 정답을 명확히 제시하세요.

# [지식 그래프 관계 데이터]:
# {graph_context}

# [질문]: {target_query}

# [추론 경로 및 최종 답변]:"""

graph_prompt = f"""당신은 지식 그래프(Knowledge Graph) 기반의 심층 추론 AI 어시스턴트입니다.
반드시 아래 제공된 [지식 그래프 관계 데이터]의 트리플렛(엔티티와 엣지)에만 근거하여 논리적 비약 없이 질문에 답변하세요.
[답변 작성 원칙]:
1. **🔍 핵심 개체 및 시작점 식별**: 질문의 대상이 되는 시작 엔티티와 이와 직접 연결된 1차 관계를 먼저 설명하세요.
2. **🔗 단계별 다중 홉(Multi-Hop) 추론 경로**: 최종 정답에 도달하기 위해 거친 지식 그래프 경로를 `[출발 노드] ──(관계)──> [중간 노드] ──(관계)──> [도착 노드]` 형태로 단계별로 명확히 보여주세요.
3. **📋 상세 근거 및 맥락 설명**: 그래프 데이터에 나타난 부서, 직급, 수치, 규정 등의 상세 속성을 결합하여 논리적으로 설명하세요.
4. **🎯 최종 결론**: 질문에 대한 명확한 핵심 정답을 정리하세요.
[지식 그래프 관계 데이터]:
{graph_context}
[질문]: {target_query}
[지식 그래프 기반 심층 답변]:"""

response = llm.invoke(graph_prompt)
clean_answer = normalize_content(response.content)
print(f"\n🎉 [Graph RAG (2-Hop) 최종 추론 답변]:\n{clean_answer}")


## 6. [Part 2] Microsoft GraphRAG 구축 원리 및 사전 인덱싱 아키텍처

Microsoft GraphRAG는 단순히 노드를 찾는 것을 넘어, 전사적 거시 질문(Global Sensemaking)에 답할 수 있는 계층적 커뮤니티 구조를 생성합니다.

### 🏗️ Microsoft GraphRAG 구축 파이프라인 (10-Stage Pipeline)

```mermaid
flowchart TD
    A[사내 원천 텍스트] --> B[1. Text Chunking] 
    B --> C[2. LLM Entity & Relationship Extraction]
    C --> D[3. Graph Merge & Finalize]
    D --> E[4. Leiden Hierarchical Clustering]
    E -->|Level 0, 1, 2| F[5. LLM Community Report Generation]
    F --> G[6. LanceDB Vector Embedding]
    G --> H[(Prebuilt Output Parquet & Vector Artifacts)]
    
    style H fill:#4a154b,stroke:#fff,stroke-width:2px,color:#fff
```

### 📂 저장된 GraphRAG 산출물 구조 (`./data/graphrag/output/`)
- `entities.parquet`: 추출된 엔티티(노드) 목록 및 설명
- `relationships.parquet`: 엔티티 간의 관계(엣지) 및 가중치
- `communities.parquet`: Leiden 계층 군집화 결과
- `community_reports.parquet`: **[핵심] 12개 커뮤니티별 LLM 심층 요약 리포트 (Global Search의 소스)**
- `lancedb/`: 엔티티 및 텍스트 청크 벡터 인덱스 (Local Search의 소스)

> 💡 **교육생 안내**: 실습 환경에는 사전 빌드된 `output/` 데이터가 준비되어 있으므로, 2분이 소요되는 인덱싱을 다시 빌드하지 않고 즉시 고성능 추론 실습을 진행할 수 있습니다.


In [ ]:
import os
import pandas as pd

GRAPHRAG_OUTPUT = "data/graphrag/output"
if not os.path.exists(GRAPHRAG_OUTPUT):
    GRAPHRAG_OUTPUT = "../data/graphrag/output"

# 1. 사전 구축된 Parquet 테이블 로드
df_entities = pd.read_parquet(os.path.join(GRAPHRAG_OUTPUT, "entities.parquet"))
df_relations = pd.read_parquet(os.path.join(GRAPHRAG_OUTPUT, "relationships.parquet"))
df_reports = pd.read_parquet(os.path.join(GRAPHRAG_OUTPUT, "community_reports.parquet"))

print("=" * 70)
print("📦 [사전 구축된 Microsoft GraphRAG 인덱스 데이터셋 검증]")
print(f"  • 추출된 엔티티 (Entities): {len(df_entities)}개")
print(f"  • 추출된 관계 (Relationships): {len(df_relations)}개")
print(f"  • 생성된 커뮤니티 리포트 (Community Reports): {len(df_reports)}개")
print("=" * 70)

# 2. 커뮤니티 요약 보고서 구조 미리보기 (상위 4개 리포트)
print("\n📑 [생성된 계층 커뮤니티 요약 리포트 목록 (Top 4)]:")
preview_cols = ["community", "title", "summary"]
for idx, row in df_reports.head(4).iterrows():
    print(f"\n[Community {row['community']}] {row['title']}")
    print(f"  -> 요약: {row['summary'][:150]}...")


## 7. [Part 2] 사전 구축된 GraphRAG 기반 Global Search vs Local Search 추론 실무

### 🧠 검색 방식별 커뮤니티 맥락 & Triplet 융합 추론 메커니즘

```mermaid
flowchart TD
    subgraph Global_Search [1. Global Search: 거시 질문 Map-Reduce]
        Q1[전사 전략 프로젝트 종합 요약 질문] --> C1[12개 Community Reports 로드]
        C1 --> M1[Map: 커뮤니티별 중간 핵심 포인트 추출]
        M1 --> R1[Reduce: 종합 전사 리포트 생성 및 인용]
    end

    subgraph Local_Search [2. Local Search: 미시 질문 서브그래프 융합]
        Q2[김철수 수석의 역할과 결재선 질문] --> V2[LanceDB 엔티티 벡터 매칭]
        V2 --> E2[엔티티 주변 1~2 Hop 서브그래프 Triplet]
        E2 --> F2[소속 커뮤니티 맥락 + 원본 텍스트 청크 융합]
        F2 --> A2[정밀 결재선/역할 답변 도출]
    end
```


In [ ]:
from rag.graphrag_tool import run_graphrag_query

GRAPHRAG_ROOT = "data/graphrag"
if not os.path.exists(GRAPHRAG_ROOT):
    GRAPHRAG_ROOT = "../data/graphrag"

# 1. Global Search 실행 (거시적 종합 질문)
global_query = "전사적으로 진행 중인 모든 전략 프로젝트들의 핵심 목표, 리스크 및 공통 의존성을 종합 요약해줘."
print("=" * 70)
print(f"🌐 [Global Search (Map-Reduce) 질의]:\n{global_query}")
print("=" * 70)
global_resp = run_graphrag_query(global_query, method="global", root_dir=GRAPHRAG_ROOT)
print(global_resp)

# 2. Local Search 실행 (미시적 엔티티 중심 질문)
local_query = "클라우드운영팀 김철수 수석의 역할과 상위 결재 승인선 및 전결 권한은 어떻게 되나요?"
print("\n" + "=" * 70)
print(f"🎯 [Local Search (Entity Neighborhood) 질의]:\n{local_query}")
print("=" * 70)
local_resp = run_graphrag_query(local_query, method="local", root_dir=GRAPHRAG_ROOT)
print(local_resp)


## 8. [Part 3] 에이전트 도구 바인딩 및 자율 질의응답 (Agentic RAG)

`01_unstructured_indexing.ipynb`의 에이전틱 RAG 설계 패턴을 지식 그래프 도구로 확장합니다.
에이전트는 사용자의 질문 의도를 분석하여:
- **거시적 전사 종합 질문** $\rightarrow$ `query_graphrag_global` 호출
- **특정 인물/부서/프로젝트 정밀 질문** $\rightarrow$ `query_graphrag_local` 호출
- **경량 빠른 관계 탐색** $\rightarrow$ `query_naive_graph_tool` 호출
을 자율적으로 결정하고 최종 응답을 완성합니다.


In [ ]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from rag.graphrag_tool import query_graphrag_global, query_graphrag_local

# 1. 경량 인메모리 지식 그래프 검색 도구 정의
@tool
def query_naive_graph_tool(seed_entity: str, max_hops: int = 2) -> str:
    """
    [인메모리 경량 지식 그래프 검색 도구]
    특정 엔티티(인물명, 팀명 등)로부터 시작하여 N-hop(1~2단계)으로 연결된
    직속 상위/하위/동료/프로젝트 관계 트리플렛을 고속으로 순회 탐색합니다.
    """
    sub, visited = multi_hop_search(G, seed_entities=[seed_entity], max_hops=max_hops)
    if not sub:
        return f"엔티티 '{seed_entity}'와 연결된 관계를 찾을 수 없습니다."
    return "\n".join([f"- [{u}] --({d['relation']})--> [{v}]" for u, v, d in sub])

# 2. 도구 툴킷 목록 등록
graph_tools = [query_naive_graph_tool, query_graphrag_global, query_graphrag_local]
tool_map = {t.name: t for t in graph_tools}

print(f"✅ 총 {len(graph_tools)}개의 Graph RAG 전용 도구가 등록되었습니다:")
for t in graph_tools:
    print(f"  • {t.name}: {t.description.strip().split('\n')[0]}")

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage
from app.utils.message_utils import normalize_content

# 1. create_agent 기반 Graph RAG 에이전트 구축
agent_system_prompt = """당신은 지식 그래프 전문 수석 어시스턴트입니다.
반드시 등록된 지식 그래프 도구를 활용하여 근거 데이터를 확보한 후 질문에 논리적 비약 없이 구체적으로 답변하세요.
- 전사적이고 포괄적인 프로젝트 현황이나 리스크를 물으면 'query_graphrag_global'을 사용하세요.
- 특정 인물, 부서, 직급, 세부 승인선 규정을 물으면 'query_graphrag_local'을 사용하세요.
- 특정 인물의 단순 상하 관계나 직속 연결선을 물으면 'query_naive_graph_tool'을 사용하세요.
"""

graph_agent = create_agent(
    model=llm,
    tools=graph_tools,
    system_prompt=agent_system_prompt,
    checkpointer=MemorySaver()
)

def run_agentic_graph_rag(user_query: str, thread_id: str = "graph_demo"):
    """
    LangChain create_agent 기반 자율 도구 라우팅 및 답변 생성 실행기
    """
    print("=" * 75)
    print(f"👤 [사용자 질문]: {user_query}")
    print("=" * 75)
    
    config = {"configurable": {"thread_id": thread_id}}
    result = graph_agent.invoke(
        {"messages": [HumanMessage(content=user_query)]},
        config=config
    )
    
    # 에이전트의 도구 선택 및 실행 로그 출력
    for m in result["messages"]:
        if hasattr(m, "tool_calls") and m.tool_calls:
            for tc in m.tool_calls:
                print(f"\n🤖 [에이전트 도구 선택]: {tc['name']}")
                print(f"   전달 인자: {tc['args']}")
        elif m.type == "tool":
            print(f"   -> 도구 실행 완료 (반환 데이터: {len(str(m.content))}자)")
            
    last_msg = result["messages"][-1]
    clean_answer = normalize_content(last_msg.content)
    print(f"\n💬 [에이전트 최종 답변]:\n{clean_answer}")
    return clean_answer

# 테스트 시나리오 1: 전사 거시 질문 (Global Search 도구 선택 유도)
run_agentic_graph_rag("전사 프로젝트들의 공통적인 예산 승인 체계를 브리핑해줘.", thread_id="test_1")

print("\n\n")
# 테스트 시나리오 2: 특정 인물/부서 미시 질문 (Local Search / Naive Graph 도구 선택 유도)
run_agentic_graph_rag("클라우드운영팀 김철수 수석의 정확한 소속 본부장과 결재 전결 한도는 얼마인가요?", thread_id="test_2")


## 9. 🎓 엔터프라이즈 Graph RAG 실무 핵심 요약 (Key Takeaways)

| 구분 | Naive Vector RAG | 기초 Graph RAG (Track 1) | Microsoft GraphRAG (Track 2) |
| :--- | :--- | :--- | :--- |
| **핵심 기술** | 단순 텍스트 청킹 + 코사인 유사도 | Pydantic Triplet 추출 + NetworkX BFS | Leiden 계층 군집화 + Map-Reduce Summaries |
| **적합한 질문** | 단일 단락에 정답이 있는 단순 사실 질의 | 2~3단계 다중 홉(Multi-hop) 조직/의존성 추적 | 전사적 거시 질문 (전체 트렌드, 리스크, 공통 패턴 요약) |
| **장점** | 구현이 매우 단순하고 빠름 | 인메모리 초경량, 명확한 관계 추론 경로 제공 | 데이터 전체를 아우르는 고차원 통찰(Sensemaking) 도출 |
| **에이전트 융합** | 단일 Vector Retriever 도구 | `query_naive_graph_tool` 도구 바인딩 | `query_graphrag_global / local` 도구 바인딩 |

> **실무 권장 아키텍처**:
> 엔터프라이즈 Agentic RAG 플랫폼에서는 어느 하나의 기술에만 의존하지 않고, **Vector RAG(비정형 문서 본문)**, **경량 DiGraph(직속 관계)**, **MS GraphRAG(전사 거시 통찰)**를 도구 툴킷으로 모두 등록한 뒤 **LangGraph 에이전트가 질문 의도에 맞추어 최적의 검색 도구를 자율 라우팅하도록 구축**하는 것이 가장 강력한 표준 아키텍처입니다.
